## Notebook Description: Agricultural Data Processing and Analysis
This Colab notebook outlines a detailed data processing pipeline for agricultural research, focusing on gas and nutrition measurements. The workflow begins with loading datasets from Google Drive, followed by robust data cleaning and standardization of identifiers using functions such as `clean_standardize_ids()`. It involves converting data types, filtering records based on criteria like functional group and batch, and aggregating numerical data (e.g., $ch4\_8h\_ml$, $ch4\_24h\_ml$, $tddm$, $dm\_percentage$, $ash\_dm$, $om\_percentage$, $pc\_percentage\_dm$, $adf\_percentage\_dm$, $ndf\_percentage\_dm$) to calculate averages and replicate counts using a custom `mean_with_replicates()` function. The processed data is then compiled and exported into several specialized CSV files, serving diverse analytical needs including training sets for a Shiny App, gas dashboards, metabolomics primary traits, and analyses for grasses breeding and Stylosanthes gene bank.

## i) Import Libraries

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import re
import pandas as pd

## ii) Data Cleaning Functions

In [ ]:
# Function to remove the first row (duplicated original column names)
def remove_first_row(df):
    return df.iloc[1:, :].copy()

# Usage:
# subset_1_information_samples = remove_first_row(subset_1_information_samples)





# Function to clean and standardize lab/sample IDs to FXX-XXXX format
def clean_lab_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'\s+', '-', regex=True)                 # Replace spaces with -
        .str.replace(r'^(F\d{2})(\d+)', r'\1-\2', regex=True) # Format FXX-XXXX
    )


# Usage:
# subset_1_information_samples['10_ciat_lab_id'] = clean_lab_ids(subset_1_information_samples['10_ciat_lab_id'])



# Function to clean and standardize Gene Bank - Breeding program IDs
def clean_standardize_ids(series):
    return (
        series
        .astype(str)
        .str.replace(r'-1$', '', regex=True)                   # Remove trailing -1
        .str.strip()                                           # Remove leading/trailing spaces
        .str.replace(r'[_\s]+', '-', regex=True)               # Replace spaces/underscores with -
        .str.replace(r'([A-Za-z])(\d)', r'\1-\2', regex=True)  # Letter followed by number
        .str.replace(r'(\d)([A-Za-z])', r'\1-\2', regex=True)  # Number followed by letter
        .str.replace(r'-+', '-', regex=True)                   # Remove repeated -
        .str.replace(r'^-|-$', '', regex=True)                 # Remove leading/trailing -
        .str.replace('ABC-', 'CIAT-', regex=False)             # Replace id's strings 'ABC' with 'CIAT'
    )

    # subset_1_information_samples['10_gene_bank_breeding_program_id'] = clean_standardize_ids(subset_1_information_samples['10_gene_bank_breeding_program_id'])


# 1.0 Shiny App Training Set

 Requested by: Khaled Al-Sham'aa, ICARDA

 Date: 2026_05_19

In [ ]:
subsets_gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_sorted_by_sql.csv')
subsets_gas.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,3,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,62.577353,4.000526,11.156690,14.2,17.82863877,135.945555,227.8476704,37.051645,65.414752
1,1,4,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,61.249843,4.225907,11.205181,15,18.29422035,133.168047,270.0095702,38.185077,63.799940
2,1,5,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,29.500225,63.019856,4.071031,10.976075,13.8,17.41685196,137.016371,249.4607864,37.088353,64.343484
3,1,6,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,30.827735,67.002386,4.285055,11.302938,13.9,16.86945515,148.741403,175.9559634,41.714068,60.152044
4,1,7,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,29.500225,66.117380,4.130032,11.233760,14,16.99063023,146.659412,149.5197103,39.832832,62.557335


In [ ]:
# Filter by functional group 'Grass'
subsets_gas_grass = subsets_gas[subsets_gas['functional_group'] == 'Grass']
subsets_gas_grass.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
39,1,44,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,26.019199,58.536668,2.367747,6.302361,9.1,10.76651796,122.759406,194.6816788,27.337315,48.347519
40,1,45,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,27.789212,61.806682,2.445451,6.425495,8.8,10.39611646,129.591169,117.6219867,25.413695,53.012554
41,1,46,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,25.576695,62.094165,2.506516,6.742543,9.8,10.85857685,130.141925,195.7779286,28.971955,48.776690
42,1,47,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,17.169131,44.186601,1.837097,7.078486,10.7,16.01953085,92.531478,89.5309162,25.377242,58.411031
43,1,48,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,20.709158,51.726628,2.360844,8.502303,11.4,16.43699461,108.342750,110.6977836,30.734726,57.941924


In [ ]:
# Filter by subset '2'
subset_2_gas_grass = subsets_gas_grass[subsets_gas_grass['subset'] == 2]
subset_2_gas_grass.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
2373,2,101,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,20.945160,58.779199,3.246500,10.094461,15.5,18.1,NaN,NaN,NaN,NaN
2374,2,102,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,20.060153,55.239172,3.089264,9.667740,15.4,18.7,117.726263,#DIV/0!,164.369645,12.535150
2375,2,103,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,20.060153,57.451689,3.169504,10.161721,15.8,18.7,122.564018,#DIV/0!,60.146889,36.042450
2472,2,206,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,19.396398,47.915741,2.967649,7.901495,15.3,17.3,102.159310,#DIV/0!,47.734635,35.291931
2473,2,207,LMF,F25-0008,Sample-8,Poales,Poaceae,Cynodon,nlemfuensis,Cynodon nlemfuensis,...,19.838901,50.039757,2.995674,8.099619,15.1,16.9,106.687844,#DIV/0!,52.077666,33.159869


In [ ]:
subset_2_gas_grass.batch.unique()

array([34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61])

In [ ]:
# Filter by batch =< 40
subset_2_gas_grass_batchs = subset_2_gas_grass[subset_2_gas_grass['batch'] >= 50]
subset_2_gas_grass_batchs.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
3870,2,1701,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,36.432778,79.886610,4.881992,12.225690,13.4,16.9,155.827612,34.19146976,43.344262,55.018936
3871,2,1702,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,36.875281,82.099127,5.125664,12.723270,13.9,16.8,155.999232,35.00485836,43.501476,55.574898
3872,2,1703,Breding,F25-2077,CIAT-PM-21-3535,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,35.105268,79.444106,5.160474,12.653738,14.7,16.9,153.594861,35.25395301,43.886070,55.745158
3873,2,1704,Breding,F25-2078,CIAT-PM-21-6106,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,35.105268,75.019073,5.090264,12.194921,14.5,17.8,148.711930,42.1835579,39.267646,61.562784
3874,2,1705,Breding,F25-2078,CIAT-PM-21-6106,Poales,Poaceae,Megathyrsus,maximus,Megathyrsus maximus,...,39.530302,82.984133,5.652833,13.170346,14.3,17.3,160.584546,36.28321101,44.216137,57.640147


In [ ]:
#subset_2_gas_grass_batchs.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/ciat_gas_for_shiny_app_and_blues.csv', index = None)

# 2.0 Gas Dashboard of Subset 4

Requested by: Alejandra Marín

Date: 2026_05_22

In [ ]:
dashboard_small = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_dashboard_small.csv')
dashboard_small.head()

,subset,id_lab,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,1,F24-3469,ABC-10647,Stylosanthes scabra,Herbaceous_legumes,15.90,18.79,48.31,67.34
1,1,F24-3470,ABC-11194,Stylosanthes hamata,Herbaceous_legumes,15.17,18.02,46.12,59.25
2,1,F24-3471,ABC-11999,Stylosanthes guianensis,Herbaceous_legumes,12.98,16.25,46.61,58.55
3,1,F24-3472,ABC-12318,Stylosanthes hamata,Herbaceous_legumes,13.99,16.96,52.27,56.88
4,1,F24-3427,ABC-1257,Stylosanthes scabra,Herbaceous_legumes,14.53,16.40,43.29,58.67


In [ ]:
# Filter data of subset 4
dashboard_small_subset_4 = dashboard_small[dashboard_small['subset'] == 4]

In [ ]:
# Order by id
def natural_sort_key(s):
    """Splits a string into alphanumeric components and converts numbers to integers for natural sorting."""
    return [int(text) if text.isdigit() else text.lower()
            for text in re.split(r'(\d+)', str(s))]

# Apply natural sort to the 'id' column
dashboard_small_subset_4_natural_sorted = dashboard_small_subset_4.sort_values(
    by='id',
    key=lambda col: col.apply(natural_sort_key)
).reset_index(drop=True)

display(dashboard_small_subset_4_natural_sorted.head())

,subset,id_lab,id,tax_name,functional_group,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,methane_intensity,tddm
0,4,F25-2640,Cayman-Br-02-1752,Urochloa interespecifico,Grass,12.99,13.71,50.17,50.22
1,4,F25-2574,CIAT-326,Desmodium scorpiurus,Herbaceous_legumes,13.86,16.66,55.45,39.09
2,4,F25-2575,CIAT-415,Vigna radiata,Herbaceous_legumes,11.22,15.38,43.25,51.39
3,4,F25-2576,CIAT-416,Vigna radiata,Herbaceous_legumes,12.38,16.84,50.26,56.45
4,4,F25-2577,CIAT-517,Macroptilium atropurpureum,Herbaceous_legumes,11.58,14.88,52.88,37.99


In [ ]:
#dashboard_small_subset_4_natural_sorted.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/subset_4_gas_dashboard_small.csv', index = None)

# 3.0 Grasses Gas Complete

Requested by: Claudia Perea

Date: 2026_05_26

In [ ]:
subsets_gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_sorted_by_sql.csv')
subsets_gas.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,3,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,62.577353,4.000526,11.156690,14.2,17.82863877,135.945555,227.8476704,37.051645,65.414752
1,1,4,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,28.172715,61.249843,4.225907,11.205181,15,18.29422035,133.168047,270.0095702,38.185077,63.799940
2,1,5,Genetic bank,F24-3416,ABC-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,...,29.500225,63.019856,4.071031,10.976075,13.8,17.41685196,137.016371,249.4607864,37.088353,64.343484
3,1,6,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,30.827735,67.002386,4.285055,11.302938,13.9,16.86945515,148.741403,175.9559634,41.714068,60.152044
4,1,7,Genetic bank,F24-3417,ABC-707,Fabales,Fabaceae,Alysicarpus,ovalifolius,Alysicarpus ovalifolius,...,29.500225,66.117380,4.130032,11.233760,14,16.99063023,146.659412,149.5197103,39.832832,62.557335


In [ ]:
# Filter by functional group 'Grass'
grasses = subsets_gas[subsets_gas['functional_group'] == 'Grass']
grasses.head()

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,net_gas_8h_ml,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,gas_ml_g_dm_incubated_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
39,1,44,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,26.019199,58.536668,2.367747,6.302361,9.1,10.76651796,122.759406,194.6816788,27.337315,48.347519
40,1,45,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,27.789212,61.806682,2.445451,6.425495,8.8,10.39611646,129.591169,117.6219867,25.413695,53.012554
41,1,46,LMF,F24-1084,LMF,Poales,Poaceae,Brachiaria,decumbens,Brachiaria decumbens,...,25.576695,62.094165,2.506516,6.742543,9.8,10.85857685,130.141925,195.7779286,28.971955,48.776690
42,1,47,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,17.169131,44.186601,1.837097,7.078486,10.7,16.01953085,92.531478,89.5309162,25.377242,58.411031
43,1,48,LMF,F24-1095,LMF,Fabales,Fabaceae,Setaria,sphacelata,Setaria sphacelata,...,20.709158,51.726628,2.360844,8.502303,11.4,16.43699461,108.342750,110.6977836,30.734726,57.941924


In [ ]:
#grasses.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/grasses.csv', index = None)

# 4.0 Primary Traits for Metabolomics

Requested by: Jenny Gallo

Date: 2026_06_03

In [ ]:
requested = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/requested_jenny.csv')
data = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/subsets_1_2_3_4_gas_dashboard_complete.csv')

In [ ]:
requested = remove_first_row(requested)
requested.head(50)

,approach,id,tax_name,functional_group,ch4_intensity_ml_g_tddm,tddm_percentage,ch4_intensity_decrease,ch4_percentage_8h,ch4_percentage_24h,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15
2,Approach 1 (a),CIAT-19213,Clitoria ternatea,Climber,39.1,63.7,39%,14.98888889,16.08929282,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Approach 2,CIAT-9434,Clitoria ternatea,Climber,46.51,51.78,33%,14.36666667,15.85424307,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Approach 2,CIAT-955,Clitoria ternatea,Climber,45.45,59.65,30%,12.82222222,14.54174742,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Approach 4,CIAT-9432,Clitoria ternatea,Climber,57.24,50.87,11%,17.36,18.90480266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Approach 4,CIAT-18447,Clitoria ternatea,Climber,53.0,55.3,18%,14.51111111,16.92415425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Approach 1 (a),CIAT-7317,Canavalia sp.,Climber,33.0,69.5,49%,14.86111111,16.36328667,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Approach 1 (a),CIAT-8719,Canavalia sp.,Climber,35.8,69.0,45%,14.66388889,16.92290927,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Approach 2,CIAT-20803,Canavalia sp.,Climber,42.61,55.70,34%,13.8,15.97697193,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,Approach 4,CIAT-19032,Canavalia sp.,Climber,65.33,43.83,6%,13.875,16.02639497,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,Approach 1 (a),CIAT-15154,Centrosema molle,Climber,38.17,61.95,41%,14,16.05586855,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
data.head()

,subset,requisitioner,no,id_lab,id,n_replicates,tax_order,family,genus,species,...,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_8h_percentage,ch4_24h_percentage,part_fact,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,1,LMF,2202,F25-0769,AMC-ICARDA-156302,3,Fabales,Fabaceae,Trifolium,repens,...,86.99,6.03,14.08,15.97,16.37,3.75,30.99,-75.10,43.67,72.00
1,1,LMF,2235,F25-0770,AMC-ICARDA-165072,3,NaN,NaN,Vicia,tenuifolia,...,86.99,6.19,14.43,14.37,18.77,2.97,31.76,NaN,56.12,56.78
2,1,LMF,2226,F25-0773,AMC-ICARDA-168125,3,NaN,NaN,Lathyrus,sylvestris,...,61.77,4.70,10.30,15.53,17.77,3.12,22.66,NaN,55.10,42.32
3,1,Genetic bank,1057,F24-3469,CIAT-10647,6,Fabales,Fabaceae,Stylosanthes,scabra,...,79.20,6.90,14.88,15.90,22.28,3.95,32.01,151.04,48.31,67.34
4,1,Genetic bank,1060,F24-3470,CIAT-11194,6,Fabales,Fabaceae,Stylosanthes,hamata,...,70.41,5.14,12.69,15.17,20.65,3.91,27.31,184.66,46.12,59.25


In [ ]:
requested.rename(columns={'gene_bank_breeding_id': 'id'}, inplace=True)
requested['id'] = clean_standardize_ids(requested['id'])
approach_id  = requested.iloc[:, :2]
approach_id.head(50)

,approach,id
2,Approach 1 (a),CIAT-19213
3,Approach 2,CIAT-9434
4,Approach 2,CIAT-955
5,Approach 4,CIAT-9432
6,Approach 4,CIAT-18447
7,Approach 1 (a),CIAT-7317
8,Approach 1 (a),CIAT-8719
9,Approach 2,CIAT-20803
10,Approach 4,CIAT-19032
11,Approach 1 (a),CIAT-15154


In [ ]:
merged_df = approach_id.merge(data, on='id')
merged_df.head(50)

,approach,id,subset,requisitioner,no,id_lab,n_replicates,tax_order,family,genus,...,net_gas_24h_ml,ch4_8h_ml,ch4_24h_ml,ch4_8h_percentage,ch4_24h_percentage,part_fact,ch4_ml_g_dm_incubaed_24h,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm
0,Approach 1 (a),CIAT-19213,1,Genetic bank,1649,F24-3501,9,Fabales,Fabaceae,Clitoria,...,71.89,7.58,11.57,14.99,18.66,4.15,24.65,194.95,39.14,63.66
1,Approach 2,CIAT-9434,2,Genetic bank,83,F24-3635,9,Fabales,Fabaceae,Clitoria,...,81.72,6.21,13.39,14.89,17.69,3.03,29.55,3.51,56.61,54.00
2,Approach 2,CIAT-955,1,Genetic bank,32,F24-3425,9,Fabales,Fabaceae,Clitoria,...,84.29,4.83,12.25,12.82,15.90,3.29,26.67,138.42,45.45,59.65
3,Approach 4,CIAT-9432,1,Genetic bank,1039,F24-3463,6,Fabales,Fabaceae,Clitoria,...,70.80,6.57,13.41,17.47,20.60,3.57,29.38,-897.64,54.34,55.33
4,Approach 4,CIAT-9432,3,Breeding,1002,F25-2635,18,Fabales,Fabaceae,Clitoria,...,84.52,7.57,13.71,15.25,17.61,2.72,28.73,100.01,60.36,48.13
5,Approach 4,CIAT-18447,1,Genetic bank,1218,F24-3484,9,Fabales,Fabaceae,Clitoria,...,80.39,6.57,13.64,14.51,20.06,3.20,29.34,361.74,53.02,55.28
6,Approach 1 (a),CIAT-7317,1,Genetic bank,185,F24-3435,36,Fabales,Fabaceae,Canavalia,...,64.66,5.20,10.57,14.86,18.14,5.01,22.85,108.22,32.97,69.48
7,Approach 1 (a),CIAT-8719,1,Genetic bank,347,F24-3453,30,Fabales,Fabaceae,Canavalia,...,65.97,5.12,10.94,14.29,18.98,4.97,23.39,239.49,34.19,69.74
8,Approach 2,CIAT-20803,1,Genetic bank,1905,F24-3512,9,Fabales,Fabaceae,Canavalia,...,68.27,4.88,10.92,13.80,18.37,3.86,23.37,344.33,42.61,55.70
9,Approach 4,CIAT-19032,2,Genetic bank,339,F24-3647,9,Fabales,Fabaceae,Canavalia,...,80.83,5.02,12.39,13.27,17.19,3.39,26.81,-2.50,45.70,59.15


In [ ]:
merged_df.columns

Index(['approach', 'id', 'subset', 'requisitioner', 'no', 'id_lab',
       'n_replicates', 'tax_order', 'family', 'genus', 'species', 'tax_name',
       'functional_group', 'set_ciat', 'batch', 'run', 'replication',
       'syrange', 'sample_weight_g', 'undigested_dm_g', 'dm_incubated',
       'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml', 'ch4_8h_ml',
       'ch4_24h_ml', 'ch4_8h_percentage', 'ch4_24h_percentage', 'part_fact',
       'ch4_ml_g_dm_incubaed_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm'],
      dtype='object')

In [ ]:
CIAT-Mulato_II	Brachiaria interespecifico
CIAT-BR02_1752	Brachiaria interespecifico
CIAT-BR15_2891	Brachiaria interespecifico
CIAT-BR19_5029	Brachiaria interespecifico


In [ ]:
primary_traits = merged_df[['approach', 'id','tax_name','functional_group', 'dm_incubated',  'digested_feed_mg', 'ch4_24h_ml']]
primary_traits.head(50)

,approach,id,tax_name,functional_group,dm_incubated,digested_feed_mg,ch4_24h_ml
0,Approach 1 (a),CIAT-19213,Clitoria ternatea,Herbaceous_legumes,469.33,298.75,11.57
1,Approach 2,CIAT-9434,Clitoria ternatea,Herbaceous_legumes,454.10,236.60,13.39
2,Approach 2,CIAT-955,Clitoria ternatea,Herbaceous_legumes,459.34,274.00,12.25
3,Approach 4,CIAT-9432,Clitoria ternatea,Herbaceous_legumes,456.56,252.63,13.41
4,Approach 4,CIAT-9432,Clitoria ternatea,Herbaceous_legumes,477.33,229.65,13.71
5,Approach 4,CIAT-18447,Clitoria ternatea,Herbaceous_legumes,464.97,257.04,13.64
6,Approach 1 (a),CIAT-7317,Canavalia sp.,Herbaceous_legumes,462.50,321.35,10.57
7,Approach 1 (a),CIAT-8719,Canavalia sp.,Herbaceous_legumes,467.83,326.25,10.94
8,Approach 2,CIAT-20803,Canavalia sp.,Herbaceous_legumes,467.17,260.21,10.92
9,Approach 4,CIAT-19032,Canavalia sp.,Herbaceous_legumes,462.32,273.47,12.39


In [ ]:
#primary_traits.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/primary_traits_to_iomicas.csv', index=None)

# 5.0 Grasses for Breeding

Requested by: Claudia Perea

Date: 2026_05_26

## 5.1 Gas Data Processing for Breeding

### 5.1.1 Gas Data Loading

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/05_grasses_for_breeding/gas_clean_subsets_1234_2026_06_09.csv')
df.head(2)


,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,...,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2,delete
0,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,0,95.43754089,34.829738,Standar,NaN,NaN,NaN,NaN,NaN,no
1,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,...,0,89.75200777,36.194699,Standar,NaN,NaN,NaN,NaN,NaN,no


### 5.1.2 Filtering and Formatting Gas Data

In [ ]:
df.functional_group.unique()

array(['Forage', 'Concentrate', 'Herbaceous_legumes', 'Grass',
       'Shrub_Trees', nan, 'Browse'], dtype=object)

In [ ]:
df.requisitioner.unique()

array(['LMF', 'Genetic_bank', 'Breeding', nan, 'Set 2', 'LMF-invivo',
       'Benchmark', 'Isabel Molina', 'Mauricio Sotelo',
       'Jacobo Arango/Alejandro Montoya',
       'Jacobo Arango/ Alejandro Montoya'], dtype=object)

In [ ]:
df1 = df[df['functional_group'] == 'Grass']

In [ ]:
df1.to_csv('grass.csv', index=None)

In [ ]:
df2 = df1[df1['requisitioner'].isin(['Breeding', 'Benchmark'])]

In [ ]:
df2.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat', 'batch',
       'run', 'replication', 'syrange', 'sample_weight_g', 'undigested_dm_g',
       'dm_incubated', 'digested_feed_mg', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'ch4_8h_ml', 'ch4_24h_ml', 'ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'gas_ml_g_dm_incubated_24h', 'part_fact',
       'ch4_ml_g_dm_incubated_24h', 'ch4_ml_g_ndf_digested_24h',
       'methane_intensity', 'tddm', 'information_remarks_1',
       'information_remarks_2', 'gas_remarks_1', 'gas_remarks_2',
       'digest_remarks_1', 'digest_remarks_2', 'delete'],
      dtype='object')

In [ ]:
df3 = df2[['subset','no','requisitioner','id_lab','id','tax_name','functional_group',
           'batch','run','replication', 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h'
       ]]
df3 = df3.copy()

cols = [ 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h']

df3[cols] = df3[cols].apply(pd.to_numeric, errors='coerce')
df3[cols] = df3[cols].round(2)

In [ ]:
df3.tail()

,subset,no,requisitioner,id_lab,id,tax_name,functional_group,batch,run,replication,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
6698,4,1144,Breeding,F25-2166,CIAT-BH-22-0528,Urochloa humidicola,Grass,65.0,2.0,2.0,32.86,86.98,177.11,45.82,2.59,13.0,14.31,25.34,55.30,0.47
6759,4,1213,Breeding,F25-2104,CIAT-BR-06-1348,Brachiaria interespecifico,Grass,71.0,3.0,1.0,42.53,95.72,205.57,54.04,2.63,13.8,14.91,30.65,56.72,0.62
6760,4,1214,Breeding,F25-2104,CIAT-BR-06-1348,Brachiaria interespecifico,Grass,71.0,3.0,2.0,38.03,92.72,199.04,69.56,3.49,13.4,14.70,29.25,42.06,0.28
6761,4,1215,Breeding,F25-2166,CIAT-BH-22-0528,Urochloa humidicola,Grass,71.0,3.0,1.0,35.53,90.22,183.54,51.32,2.80,14.1,15.19,27.88,54.33,0.41
6762,4,1216,Breeding,F25-2166,CIAT-BH-22-0528,Urochloa humidicola,Grass,71.0,3.0,2.0,35.03,88.22,179.44,38.84,2.16,13.4,14.79,26.53,68.32,0.77


In [ ]:
#df3.to_csv('breeding.csv', index=None)

In [ ]:
#df3.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/gas_breeding_complete.csv', index=None)

### 5.1.3 Average Gas Values

In [ ]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_name', 'functional_group', 'batch',
       'run', 'replication']
numeric_columns = [ 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h']

for df in [df3]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_gas'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [ ]:
df4 = mean_with_replicates(df3, category_columns, numeric_columns)
df4.tail(2)

/tmp/ipykernel_8580/1161671181.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_8580/1161671181.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_8580/1161671181.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


,id_lab,subset,no,requisitioner,id,tax_name,functional_group,batch,run,replication,...,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas
179,F25-2627,3,978,Benchmark,BR-02-1752-Cayman-Exc,Urochloa interespecifico,Grass,55.0,1.0,1.0,...,92.91,191.99,55.26,2.88,60.35,36.13,68.69,124.90,604.43,15
180,F25-2628,3,981,Benchmark,BR-06-423-Cayman-Exc,Urochloa interespecifico,Grass,55.0,1.0,1.0,...,91.17,188.56,55.81,2.97,58.46,36.00,66.75,119.16,576.23,15


In [ ]:
df4

,id_lab,subset,no,requisitioner,id,tax_name,functional_group,batch,run,replication,...,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas
0,F24-3582,1,226,Breeding,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,10.0,1.0,1.0,...,74.98,156.06,67.39,4.32,13.98,14.61,22.79,33.83,100.28,9
1,F24-3583,1,229,Breeding,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,10.0,1.0,1.0,...,76.60,157.75,63.97,4.06,13.79,14.51,22.88,35.77,116.14,9
2,F24-3584,1,232,Breeding,CIAT-BR-06-0423,Brachiaria interespecifico,Grass,10.0,1.0,1.0,...,73.82,152.46,64.27,4.22,14.61,15.14,23.12,35.98,108.27,8
3,F24-3585,1,235,Breeding,CIAT-BR-09-1232,Brachiaria interespecifico,Grass,10.0,1.0,1.0,...,81.29,167.08,66.64,3.99,13.76,14.37,24.02,36.04,108.20,8
4,F24-3586,1,238,Breeding,CIAT-BR-12-4951,Brachiaria interespecifico,Grass,10.0,1.0,1.0,...,75.13,156.86,65.93,4.21,14.34,15.02,23.59,35.76,103.45,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
176,F25-2624,3,969,Benchmark,CIAT-6294-Marandu-Exc,Urochloa interespecifico,Grass,55.0,1.0,1.0,...,87.48,183.06,51.82,2.86,53.96,31.61,58.22,110.66,528.96,14
177,F25-2625,3,1080,Benchmark,CIAT-36087-MulatoII-Exc,Urochloa interespecifico,Grass,55.0,2.0,1.0,...,94.65,195.52,58.75,3.01,94.65,51.47,100.78,171.43,829.90,6
178,F25-2626,3,975,Benchmark,CIAT-606-Basilisk-Exc,Urochloa interespecifico,Grass,55.0,1.0,1.0,...,99.31,204.80,56.39,2.76,63.81,36.74,74.39,130.67,633.62,15
179,F25-2627,3,978,Benchmark,BR-02-1752-Cayman-Exc,Urochloa interespecifico,Grass,55.0,1.0,1.0,...,92.91,191.99,55.26,2.88,60.35,36.13,68.69,124.90,604.43,15


In [ ]:
#df4.to_csv('breeding_average.csv', index=None)

In [ ]:
#df4.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/gas_breeding_average.csv', index=None)

## 5.2. Nutrition Data Processing for Breeding

### 5.2.1 Nutrition Data Loading

In [ ]:
nu = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/05_grasses_for_breeding/nutrition_complete_1234_2026_06_09.csv')
nu.head(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic_bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,91.930807,12.725691,87.274309,33.010522,28.489614,45.210329
1,1,2.0,Genetic_bank,F24-3416,CIAT-705,Fabales,Fabaceae,Indigofera,suffruticosa,Indigofera suffruticosa,Herbaceous_legumes,1,92.010000,12.629062,87.370938,33.010522,27.612657,45.235119


### 5.2.2 Filtering and Formatting Nutrition Data

In [ ]:
nu.functional_group.unique()

array(['Herbaceous_legumes', 'Shrub_Trees', 'Grass', 'Browse', 'Forage',
       'Concentrate', nan], dtype=object)

In [ ]:
nu.requisitioner.unique()

array(['Genetic_bank', 'Breeding', 'LMF', 'LMF-invivo', 'Benchmark',
       'Isabel Molina', 'Mauricio Sotelo',
       'Jacobo Arango/Alejandro Montoya',
       'Jacobo Arango/ Alejandro Montoya'], dtype=object)

In [ ]:
nu1 = nu[nu['functional_group'] == 'Grass']

In [ ]:
nu2 = nu1[nu1['requisitioner'].isin(['Breeding', 'Benchmark'])]
nu2.tail(2)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,functional_group,set_ciat,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
1910,4,NaN,Breeding,F25-2166,CIAT-BH-22-0528,Poales,Poaceae,Urochloa,humidicola,Urochloa humidicola,Grass,30,98.20,12.48,87.52,8.80,36.63,73.37
1911,4,NaN,Breeding,F25-2166,CIAT-BH-22-0528,Poales,Poaceae,Urochloa,humidicola,Urochloa humidicola,Grass,30,98.22,15.08,84.92,8.73,36.96,73.74


In [ ]:
cols = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']
nu2[cols] = nu2[cols].round(2)

/tmp/ipykernel_8580/1946393332.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nu2[cols] = nu2[cols].round(2)


In [ ]:
#nu2.to_csv('nutrition.csv', index=None)

In [ ]:
#nu2.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/nutrition_breeding_complete.csv', index=None)

In [ ]:
nu2.columns

Index(['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'functional_group', 'set_ciat',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm'],
      dtype='object')

In [ ]:
nu3 = nu2[['subset','no','requisitioner','id_lab','id','tax_name','functional_group','dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']]
nu3 = nu3.copy()

cols = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']
nu3[cols] = nu3[cols].round(2)

In [ ]:
nu3

,subset,no,requisitioner,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
52,1,51.0,Breeding,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,95.98,15.64,84.36,10.92,21.20,55.59
53,1,52.0,Breeding,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,96.07,15.76,84.24,10.92,21.09,55.23
54,1,53.0,Breeding,F24-3583,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,96.97,15.14,84.86,9.39,21.27,54.83
55,1,54.0,Breeding,F24-3583,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,97.12,14.90,85.10,9.39,22.40,56.70
56,1,55.0,Breeding,F24-3584,CIAT-BR-06-0423,Brachiaria interespecifico,Grass,96.56,15.97,84.03,11.01,22.15,57.45
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1521,3,212.0,Benchmark,F25-2628,BR-06-423-Cayman-Exc,Urochloa interespecifico,Grass,96.77,10.43,89.57,13.82,29.08,61.44
1908,4,NaN,Breeding,F25-2104,CIAT-BR-06-1348,Brachiaria interespecifico,Grass,93.02,14.27,85.73,12.27,25.43,59.24
1909,4,NaN,Breeding,F25-2104,CIAT-BR-06-1348,Brachiaria interespecifico,Grass,93.16,14.68,85.32,12.18,25.85,60.28
1910,4,NaN,Breeding,F25-2166,CIAT-BH-22-0528,Urochloa humidicola,Grass,98.20,12.48,87.52,8.80,36.63,73.37


### 5.2.3 Average Nutrition Values

In [ ]:
#Columns processing

category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_name', 'functional_group']
numeric_columns = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm','adf_percentage_dm', 'ndf_percentage_dm']

for df in [nu3]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id_lab']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id_lab'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id_lab'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_nutrition'] = (
        df.groupby([ 'id_lab'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [ ]:
nu4 = mean_with_replicates(nu3, category_columns, numeric_columns)
nu4.tail(2)

/tmp/ipykernel_8580/1194620962.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])[cat_cols]
/tmp/ipykernel_8580/1194620962.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id_lab'])[numeric_columns]
/tmp/ipykernel_8580/1194620962.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id_lab'])


,id_lab,subset,no,requisitioner,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
179,F25-2627,3,209.0,Benchmark,BR-02-1752-Cayman-Exc,Urochloa interespecifico,Grass,96.64,17.21,82.79,10.93,29.65,59.76,2
180,F25-2628,3,211.0,Benchmark,BR-06-423-Cayman-Exc,Urochloa interespecifico,Grass,96.67,10.42,89.57,13.86,29.45,61.60,2


In [ ]:
nu4.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/nutrition_breeding_average.csv', index=None)

## 5.3 Compilation of Gas and Nutrition Data

In [ ]:
df4.columns

Index(['id_lab', 'subset', 'no', 'requisitioner', 'id', 'tax_name',
       'functional_group', 'batch', 'run', 'replication', 'net_gas_8h_ml',
       'net_gas_24h_ml', 'gas_ml_g_dm_incubated_24h', 'tddm', 'part_fact',
       'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h',
       'ch4_ml_g_dm_incubated_24h', 'methane_intensity',
       'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas'],
      dtype='object')

In [ ]:
df5 = df4[['id_lab','net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas']]

In [ ]:
compiled = nu4.merge(df5, on='id_lab', how='left')

In [ ]:
#compiled.to_csv('compiled.csv', index=None)

In [ ]:
# Save the compiled gas and nutrition breeding average data
#compiled.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/05_grasses_for_breeding/compilated_gas_nutrition_breeding_average.csv', index=None)

## 5.4 Dataset for BLUES and BLUPs

In [ ]:
df3.head(2) # Filtered dataset for Breeding

,subset,no,requisitioner,id_lab,id,tax_name,functional_group,batch,run,replication,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
364,1,226,Breeding,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,10.0,1.0,1.0,36.65,73.60,153.24,66.86,4.36,13.6,14.35,21.99,32.90,98.77
365,1,227,Breeding,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,10.0,1.0,2.0,36.65,70.51,146.73,65.35,4.45,13.7,14.32,21.02,32.16,101.24


In [ ]:
batches = df3.batch.unique()
batches.dtype

CategoricalDtype(categories=[10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 23.0, 24.0, 25.0, 40.0,
                  41.0, 42.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0,
                  54.0, 55.0, 56.0, 57.0, 59.0, 60.0, 61.0, 64.0, 65.0, 71.0],
, ordered=False, categories_dtype=float64)

In [ ]:
star = df[df['id_lab'] == 'F25-0008']
star.batch.unique()

array([34., 35., 36., 37., 38., 39., 40., 41., 42., 43., 44., 45., 46.,
       47., 48., 49., 50., 51., 52., 53., 54., 55., 56., 57., 59., 60.,
       61., 58., 69., 62., 63., 70., 64., 65., 71.])

In [ ]:
star.batch.unique()

array([34., 35., 36., 37., 38., 39., 40., 41., 42., 43., 44., 45., 46.,
       47., 48., 49., 50., 51., 52., 53., 54., 55., 56., 57., 59., 60.,
       61., 58., 69., 62., 63., 70., 64., 65., 71.])

In [ ]:
star_in_batch = star[star["batch"].isin(batches)]
star_in_batch.batch.unique()

array([40., 41., 42., 46., 47., 48., 49., 50., 51., 52., 53., 54., 55.,
       56., 57., 59., 60., 61., 64., 65., 71.])

In [ ]:
star3 = star_in_batch[['subset','no','requisitioner','id_lab','id','tax_name','functional_group',
           'batch','run','replication', 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h'
       ]]
star3 = star3.copy()

cols = [ 'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h']

star3[cols] = star3[cols].apply(pd.to_numeric, errors='coerce')
star3[cols] = star3[cols].round(2)

In [ ]:
star3.batch.unique()

array([40., 41., 42., 46., 47., 48., 49., 50., 51., 52., 53., 54., 55.,
       56., 57., 59., 60., 61., 64., 65., 71.])

In [ ]:
df3.batch.unique()

[10.0, 11.0, 12.0, 13.0, 14.0, ..., 47.0, 48.0, 64.0, 65.0, 71.0]
Length: 30
Categories (30, float64): [10.0, 11.0, 12.0, 13.0, ..., 61.0, 64.0, 65.0, 71.0]

In [ ]:
breeding_and_star = pd.concat([df3, star3])

In [ ]:
#breeding_and_star.to_csv('gas_breeding_plus_stargrass.csv', index = None)

# 6.0 Stylosanthes Gene Bank

Requested by: Juan José González

Date: 2026_05_25

## 6.1 Data Loading for Stylosanthes Gene Bank

In [ ]:
requested = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/06_stylosanthes_gene_bank/data_requested_juan_jose.csv')
gas = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/06_stylosanthes_gene_bank/gas_clean_subsets_1234_2026_06_09.csv')
nu = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/06_stylosanthes_gene_bank/nutrition_complete_1234_2026_06_09.csv')

## 6.2 Filtering and Formatting Stylosanthes Gene Bank Data

In [ ]:
requested['id'] = clean_standardize_ids(requested['id'])

In [ ]:
category_columns = ['subset', 'no', 'requisitioner', 'id_lab', 'id', 'tax_order', 'family',
       'genus', 'species', 'tax_name', 'set_ciat','functional_group','batch','run','replication','syrange']
numeric_columns = ['net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h']

for df in [gas, nu]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
cols = [
    'net_gas_8h_ml','net_gas_24h_ml',
           'gas_ml_g_dm_incubated_24h',  'tddm','part_fact','ch4_percentage_in_gas_8h',
       'ch4_percentage_in_gas_24h', 'ch4_ml_g_dm_incubated_24h', 'methane_intensity', 'ch4_ml_g_ndf_digested_24h'
]

gas2 = gas[['id'] + cols].copy()


gas2[cols] = gas2[cols].round(2)


gas2.head(2)

,id,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
0,Hohenheimer-Heustandard,22.72,47.16,206.56,34.83,1.69,14.9,16.09,33.24,95.44,0.0
1,Hohenheimer-Heustandard,21.83,45.84,200.59,36.19,1.80,15.2,16.20,32.49,89.75,0.0


In [ ]:
nu2 = nu[['subset', 'no', 'requisitioner', 'id_lab', 'id', 'functional_group',
       'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']]
nu2.round(2)
cols = ['dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']
nu2[cols] = nu2[cols].round(2)
nu2.head(2)

/tmp/ipykernel_8580/3026070774.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nu2[cols] = nu2[cols].round(2)


,subset,no,requisitioner,id_lab,id,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,1,1.0,Genetic_bank,F24-3416,CIAT-705,Herbaceous_legumes,91.93,12.73,87.27,33.01,28.49,45.21
1,1,2.0,Genetic_bank,F24-3416,CIAT-705,Herbaceous_legumes,92.01,12.63,87.37,33.01,27.61,45.24


## 6.3 Compilation of Stylosanthes Gene Bank Data

In [ ]:
requested_gas = requested.merge(gas2, on='id', how='left')

In [ ]:
requested_gas

,gender,species,id,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
0,Stylosanthes,guianensis,CIAT-11313,43.05,92.68,203.13,45.97,2.26,12.80,14.57,29.59,64.37,293.67
1,Stylosanthes,guianensis,CIAT-11313,32.05,73.18,158.73,48.33,3.04,13.10,14.06,22.31,46.16,212.81
2,Stylosanthes,guianensis,CIAT-11313,33.55,76.68,166.36,49.15,2.95,12.20,13.44,22.35,45.48,209.65
3,Stylosanthes,guianensis,CIAT-11313,39.51,92.35,202.42,57.24,2.83,12.80,14.46,29.27,51.14,233.30
4,Stylosanthes,guianensis,CIAT-11313,29.01,76.35,165.62,32.84,1.98,12.40,13.39,22.18,67.53,311.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
423,Stylosanthes,scabra,CIAT-12710,36.71,77.23,168.29,51.47,3.06,77.23,44.48,74.85,145.42,667.34
424,Stylosanthes,scabra,CIAT-12710,36.71,77.23,168.29,51.47,3.06,77.23,44.48,74.85,145.42,667.34
425,Stylosanthes,scabra,CIAT-12710,37.05,82.40,181.15,45.48,2.51,82.40,45.20,81.87,180.02,818.86
426,Stylosanthes,scabra,CIAT-12710,37.05,74.40,162.13,49.07,3.03,74.40,44.43,72.03,146.78,673.58


In [ ]:
requested_nutrition = requested.merge(nu2, on='id', how='left')

In [ ]:
requested_gas

,gender,species,id,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
0,Stylosanthes,guianensis,CIAT-11313,43.05,92.68,203.13,45.97,2.26,12.80,14.57,29.59,64.37,293.67
1,Stylosanthes,guianensis,CIAT-11313,32.05,73.18,158.73,48.33,3.04,13.10,14.06,22.31,46.16,212.81
2,Stylosanthes,guianensis,CIAT-11313,33.55,76.68,166.36,49.15,2.95,12.20,13.44,22.35,45.48,209.65
3,Stylosanthes,guianensis,CIAT-11313,39.51,92.35,202.42,57.24,2.83,12.80,14.46,29.27,51.14,233.30
4,Stylosanthes,guianensis,CIAT-11313,29.01,76.35,165.62,32.84,1.98,12.40,13.39,22.18,67.53,311.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...
423,Stylosanthes,scabra,CIAT-12710,36.71,77.23,168.29,51.47,3.06,77.23,44.48,74.85,145.42,667.34
424,Stylosanthes,scabra,CIAT-12710,36.71,77.23,168.29,51.47,3.06,77.23,44.48,74.85,145.42,667.34
425,Stylosanthes,scabra,CIAT-12710,37.05,82.40,181.15,45.48,2.51,82.40,45.20,81.87,180.02,818.86
426,Stylosanthes,scabra,CIAT-12710,37.05,74.40,162.13,49.07,3.03,74.40,44.43,72.03,146.78,673.58


In [ ]:
requested_nutrition

,gender,species,id,subset,no,requisitioner,id_lab,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,Stylosanthes,guianensis,CIAT-11313,3,175.0,Genetic_bank,F24-3559,Herbaceous_legumes,92.11,11.55,88.45,15.09,38.49,55.73
1,Stylosanthes,guianensis,CIAT-11313,3,176.0,Genetic_bank,F24-3559,Herbaceous_legumes,92.26,11.52,88.48,15.09,39.16,56.20
2,Stylosanthes,guianensis,CIAT-11417,3,177.0,Genetic_bank,F24-3560,Herbaceous_legumes,91.48,11.93,88.07,14.94,41.32,57.71
3,Stylosanthes,guianensis,CIAT-11417,3,178.0,Genetic_bank,F24-3560,Herbaceous_legumes,91.52,12.19,87.81,14.94,41.63,56.74
4,Stylosanthes,guianensis,CIAT-12311,3,247.0,Genetic_bank,F24-3572,Herbaceous_legumes,91.67,15.02,84.98,16.63,35.04,52.68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Stylosanthes,scabra,CIAT-11820,3,190.0,Genetic_bank,F24-3566,Herbaceous_legumes,91.89,12.99,87.01,16.89,31.14,52.56
86,Stylosanthes,scabra,CIAT-12485,3,257.0,Genetic_bank,F24-3577,Herbaceous_legumes,94.06,13.11,86.89,14.59,30.83,56.54
87,Stylosanthes,scabra,CIAT-12485,3,258.0,Genetic_bank,F24-3577,Herbaceous_legumes,93.84,13.58,86.42,14.59,30.72,55.03
88,Stylosanthes,scabra,CIAT-12710,3,263.0,Genetic_bank,F24-3580,Herbaceous_legumes,91.68,10.75,89.25,11.50,38.96,58.26


In [ ]:
#requested_gas.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/06_stylosanthes_gene_bank/stylosanthes_gas_genebank.csv', index =None)

In [ ]:
#requested_nutrition.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/06_stylosanthes_gene_bank/stylosanthes_nutrition_genebank.csv', index = None)

# 7.0  NIRs Compilation

---

Requested by: Juan Andrés

Date: 2026_06

## 7.1 Data load

In [ ]:
gas_av = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/gas_clean_average_subsets_1234_2026_06_09.csv')
nu_av = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/nutrition_average_1234_2026_06_09.csv')

ggb = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/grass_genebank.csv')
gt = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/total_grass.csv')
leg = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/07_nirs_compilation/legumes.csv')


In [ ]:
gt

,position,id_lab,id,ot_lab,tax_name,functional_group,400,400.5,401,401.5,...,2495,2495.5,2496,2496.5,2497,2497.5,2498,2498.5,2499,2499.5
0,1.0,F243582,ABC-BR02_1752,OT-035 (2024),Brachiaria interespecifico,Grasses,0.869096,0.869367,0.869572,0.869713,...,0.595881,0.595940,0.595980,0.596001,0.596002,0.595982,0.595943,0.595884,0.595806,0.595705
1,2.0,F243583,ABC-BR02_1794,OT-035 (2024),Brachiaria interespecifico,Grasses,0.870214,0.870589,0.870905,0.871167,...,0.600557,0.600603,0.600633,0.600645,0.600641,0.600622,0.600587,0.600536,0.600469,0.600380
2,3.0,F243584,ABC-BR06_0423,OT-035 (2024),Brachiaria interespecifico,Grasses,0.868262,0.868470,0.868617,0.868706,...,0.585142,0.585161,0.585165,0.585154,0.585130,0.585091,0.585039,0.584971,0.584887,0.584782
3,4.0,F243585,ABC-BR09_1232,OT-035 (2024),Brachiaria interespecifico,Grasses,0.891681,0.892130,0.892507,0.892815,...,0.610970,0.610995,0.611003,0.610993,0.610966,0.610922,0.610861,0.610784,0.610689,0.610572
4,5.0,F243586,ABC-BR12_4951,OT-035 (2024),Brachiaria interespecifico,Grasses,0.862314,0.862796,0.863215,0.863575,...,0.588970,0.589010,0.589031,0.589032,0.589011,0.588969,0.588907,0.588824,0.588721,0.588595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,NaN,F252624,CIAT_6294_Marandu_Exc,OT-032 (2025) Benchmark,Urochloa interespecifico,grasses,0.853588,0.854769,0.855888,0.856945,...,0.599954,0.599976,0.599982,0.599969,0.599936,0.599885,0.599812,0.599718,0.599603,0.599460
182,NaN,F252625,CIAT_36087_MulatoII_Exc,OT-032 (2025) Benchmark,Urochloa interespecifico,grasses,0.877555,0.878728,0.879831,0.880866,...,0.623623,0.623638,0.623637,0.623619,0.623584,0.623529,0.623455,0.623360,0.623244,0.623102
183,NaN,F252626,CIAT_606_Basilisk_Exc,OT-032 (2025) Benchmark,Urochloa interespecifico,grasses,0.867927,0.869085,0.870174,0.871197,...,0.633598,0.633619,0.633622,0.633606,0.633573,0.633521,0.633452,0.633364,0.633258,0.633128
184,NaN,F252627,BR02_1752_Cayman_Exc,OT-032 (2025) Benchmark,Urochloa interespecifico,grasses,0.868316,0.868839,0.869309,0.869729,...,0.581532,0.581545,0.581545,0.581532,0.581504,0.581463,0.581405,0.581330,0.581236,0.581119


In [ ]:
nu_av.head(5)

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,n_replicates_nutrition
0,Dieta-1_Exp1,Dieta-1-Exp-1,3,193.0,LMF-invivo,Dieta 1_Exp1,NaN,89.00,9.50,89.53,17.68,35.48,71.52,2
1,Dieta-1_Exp2,Dieta-1-Exp-2,3,235.0,LMF-invivo,Dieta 1_Exp2,NaN,89.00,9.50,89.53,0.00,0.00,0.00,2
2,Dieta-2_Exp1,Dieta-2-Exp-1,3,195.0,LMF-invivo,Dieta 2_Exp1,NaN,89.00,9.10,89.58,13.79,33.91,67.86,2
3,Dieta-2_Exp2,Dieta-2-Exp-2,3,237.0,LMF-invivo,Dieta 2_Exp2,NaN,89.00,9.10,89.58,0.00,0.00,0.00,2
4,F24-3416,CIAT-705,1,1.0,Genetic_bank,Indigofera suffruticosa,Herbaceous_legumes,91.97,12.68,87.32,33.01,28.05,45.22,2


## 7.2 Filtering and formatting

In [ ]:
gt['id_lab'] = clean_lab_ids(gt['id_lab'])
ggb['id_lab'] = clean_lab_ids(ggb['id_lab'])
leg['id_lab'] = clean_lab_ids(leg['id_lab'])

gt['id'] = clean_standardize_ids(gt['id'])
ggb['id'] = clean_standardize_ids(ggb['id'])
leg['id'] = clean_standardize_ids(leg['id'])

## 7.3 Compilated average values for Gas and Nutrition





In [ ]:
gas_av.columns

Index(['id_lab', 'id', 'subset', 'no', 'requisitioner', 'tax_name',
       'functional_group', 'net_gas_8h_ml', 'net_gas_24h_ml',
       'gas_ml_g_dm_incubated_24h', 'tddm', 'part_fact',
       'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h',
       'ch4_ml_g_dm_incubated_24h', 'methane_intensity',
       'ch4_ml_g_ndf_digested_24h', 'n_replicates_gas'],
      dtype='object')

In [ ]:
nu_av.columns

Index(['id_lab', 'id', 'subset', 'no', 'requisitioner', 'tax_name',
       'functional_group', 'dm_percentage', 'ash_dm', 'om_percentage',
       'pc_percentage_dm', 'adf_percentage_dm', 'ndf_percentage_dm',
       'n_replicates_nutrition'],
      dtype='object')

In [ ]:
nu_av2 = nu_av[['id_lab', 'id', 'tax_name', 'functional_group', 'dm_percentage', 'ash_dm', 'om_percentage', 'pc_percentage_dm',
       'adf_percentage_dm', 'ndf_percentage_dm']]
nu_av2.head(2)

,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm
0,Dieta-1_Exp1,Dieta-1-Exp-1,Dieta 1_Exp1,NaN,89.0,9.5,89.53,17.68,35.48,71.52
1,Dieta-1_Exp2,Dieta-1-Exp-2,Dieta 1_Exp2,NaN,89.0,9.5,89.53,0.00,0.00,0.00


In [ ]:
gas_av2 = gas_av[['id_lab','net_gas_8h_ml', 'net_gas_24h_ml',
       'gas_ml_g_dm_incubated_24h', 'tddm', 'part_fact',
       'ch4_percentage_in_gas_8h', 'ch4_percentage_in_gas_24h',
       'ch4_ml_g_dm_incubated_24h', 'methane_intensity',
       'ch4_ml_g_ndf_digested_24h']]
gas_av2.head(2)

,id_lab,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
0,Dieta-1_Exp1,30.67,67.46,149.92,34.90,2.31,14.42,14.69,21.82,71.84,321.88
1,Dieta-1_Exp2,28.54,58.34,131.09,35.74,2.56,42.80,28.57,39.10,122.89,546.98


In [ ]:
compiled = nu_av2.merge(gas_av2, on='id_lab', how='left')
compiled.head(2)

,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
0,Dieta-1_Exp1,Dieta-1-Exp-1,Dieta 1_Exp1,NaN,89.0,9.5,89.53,17.68,35.48,71.52,30.67,67.46,149.92,34.90,2.31,14.42,14.69,21.82,71.84,321.88
1,Dieta-1_Exp2,Dieta-1-Exp-2,Dieta 1_Exp2,NaN,89.0,9.5,89.53,0.00,0.00,0.00,28.54,58.34,131.09,35.74,2.56,42.80,28.57,39.10,122.89,546.98


## 7.4 Traits and NIRs data frame creation

### 7.4.1 Grasses total


In [ ]:
gt.columns

Index(['position', 'id_lab', 'id', 'ot_lab', 'tax_name', 'functional_group',
       '400', '400.5', '401', '401.5',
       ...
       '2495', '2495.5', '2496', '2496.5', '2497', '2497.5', '2498', '2498.5',
       '2499', '2499.5'],
      dtype='object', length=4206)

In [ ]:
requested_gt = gt.id_lab.unique()
print('Requested length:',len(requested_gt))
print(requested_gt)

Requested length: 186
['F24-3582' 'F24-3583' 'F24-3584' 'F24-3585' 'F24-3586' 'F24-3587'
 'F24-3588' 'F24-3589' 'F24-3590' 'F24-3591' 'F24-3592' 'F24-3593'
 'F24-3594' 'F24-3595' 'F24-3596' 'F24-3597' 'F24-3598' 'F24-3599'
 'F24-3600' 'F24-3601' 'F24-3602' 'F24-3603' 'F24-3604' 'F24-3605'
 'F24-3606' 'F24-3607' 'F24-3608' 'F24-3609' 'F24-3610' 'F24-3611'
 'F25-0990' 'F25-0991' 'F25-0992' 'F25-0993' 'F25-0994' 'F25-0995'
 'F25-0996' 'F25-0997' 'F25-0998' 'F25-0999' 'F25-1000' 'F25-1001'
 'F25-1002' 'F25-1003' 'F25-1004' 'F25-1005' 'F25-1006' 'F25-1007'
 'F25-1008' 'F25-1009' 'F25-1010' 'F25-1011' 'F25-1012' 'F25-1013'
 'F25-1014' 'F25-1701' 'F25-1702' 'F25-1703' 'F25-1704' 'F25-1705'
 'F25-1706' 'F25-1707' 'F25-1708' 'F25-1709' 'F25-1710' 'F25-1711'
 'F25-1712' 'F25-1713' 'F25-1714' 'F25-1715' 'F25-1716' 'F25-1717'
 'F25-1718' 'F25-1719' 'F25-1720' 'F25-1721' 'F25-1722' 'F25-1723'
 'F25-1724' 'F25-1725' 'F25-1726' 'F25-1727' 'F25-2077' 'F25-2078'
 'F25-2079' 'F25-2080' 'F25-2081' 'F25-2

In [ ]:
gt_2 = gt.drop(['position', 'id', 'ot_lab', 'tax_name', 'functional_group'], axis=1)
gt_2.head(2)

,id_lab,400,400.5,401,401.5,402,402.5,403,403.5,404,...,2495,2495.5,2496,2496.5,2497,2497.5,2498,2498.5,2499,2499.5
0,F24-3582,0.869096,0.869367,0.869572,0.869713,0.869797,0.869827,0.869811,0.869752,0.869651,...,0.595881,0.595940,0.595980,0.596001,0.596002,0.595982,0.595943,0.595884,0.595806,0.595705
1,F24-3583,0.870214,0.870589,0.870905,0.871167,0.871380,0.871549,0.871680,0.871775,0.871833,...,0.600557,0.600603,0.600633,0.600645,0.600641,0.600622,0.600587,0.600536,0.600469,0.600380


In [ ]:
requested_compiled = compiled[compiled['id_lab'].isin(requested_gt)]
print('requested_compiled lenght:', len(requested_compiled))
requested_compiled.head(2)

requested_compiled lenght: 186


,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
170,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,96.03,15.70,84.30,10.92,21.14,55.41,37.05,74.98,156.06,67.39,4.32,13.98,14.61,22.79,33.83,100.28
171,F24-3583,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,97.05,15.02,84.98,9.39,21.84,55.76,38.62,76.60,157.75,63.97,4.06,13.79,14.51,22.88,35.77,116.14


In [ ]:
gt_traits_nirs = requested_compiled.merge(gt_2, on='id_lab', how='left')
print('gt_traits_nirs lenght:', len(gt_traits_nirs))
gt_traits_nirs.head(2)

gt_traits_nirs lenght: 186


,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,...,2495,2495.5,2496,2496.5,2497,2497.5,2498,2498.5,2499,2499.5
0,F24-3582,CIAT-BR-02-1752,Brachiaria interespecifico,Grass,96.03,15.70,84.30,10.92,21.14,55.41,...,0.595881,0.595940,0.595980,0.596001,0.596002,0.595982,0.595943,0.595884,0.595806,0.595705
1,F24-3583,CIAT-BR-02-1794,Brachiaria interespecifico,Grass,97.05,15.02,84.98,9.39,21.84,55.76,...,0.600557,0.600603,0.600633,0.600645,0.600641,0.600622,0.600587,0.600536,0.600469,0.600380


In [ ]:
#gt_traits_nirs.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/07_nirs_compilation/total_grass_nirs.csv', index = None)

### 7.4.2 Grasses for Gene Bank

In [ ]:
ggb_list = ggb.id_lab.unique()
print('Requested list length:',len(ggb_list))

compiled_list = compiled.id_lab.unique()
print('Compiled list length:',len(compiled_list))

commons = set(ggb_list) & set(compiled_list)
print('Commons list length:',len(commons))
print(commons)

Requested list length: 65
Compiled list length: 668
Commons list length: 0
set()


According with this comparison, no samples are shared between requested and compiled lists

### 7.4.3 Legumes

In [ ]:
leg.head(1)

,position,id_lab,id,ot_lab,tax_name,functional_group,Unnamed: 6,400,400.5,401,...,2495.5,2496,2496.5,2497,2497.5,2498,2498.5,2499,2499.5,Unnamed: 4207
0,1.0,F24-3416,CIAT-705,OT-033 (2024),Indigofera suffruticosa,Herbaceous,NaN,0.983552,0.985116,0.986627,...,0.59227,0.592244,0.592214,0.592179,0.592138,0.592091,0.592035,0.59197,0.591889,NaN


In [ ]:
leg_2 = leg.drop(['position', 'id', 'ot_lab', 'tax_name', 'functional_group'], axis=1)
leg_2.head(1)

,id_lab,Unnamed: 6,400,400.5,401,401.5,402,402.5,403,403.5,...,2495.5,2496,2496.5,2497,2497.5,2498,2498.5,2499,2499.5,Unnamed: 4207
0,F24-3416,NaN,0.983552,0.985116,0.986627,0.988091,0.989511,0.990893,0.992242,0.993557,...,0.59227,0.592244,0.592214,0.592179,0.592138,0.592091,0.592035,0.59197,0.591889,NaN


In [ ]:
requested_list = leg_2.id_lab.unique()
print('Requested list length:',len(requested_list))

compiled_list = compiled.id_lab.unique()
print('Compiled list length:',len(compiled_list))

commons = set(requested_list) & set(compiled_list)
print('Commons list length:',len(commons))
print(commons)

Requested list length: 462
Compiled list length: 668
Commons list length: 461
{'F24-3429', 'F25-1654', 'F24-3481', 'F25-1769', 'F25-1767', 'F25-1669', 'F24-3433', 'F24-3624', 'F25-2603', 'F24-3475', 'F24-3461', 'F25-0981', 'F25-1660', 'F24-3657', 'F25-2633', 'F25-2582', 'F24-3638', 'F24-3536', 'F25-0935', 'F25-1773', 'F25-1738', 'F25-1743', 'F25-1729', 'F25-1655', 'F25-1659', 'F25-1760', 'F24-3487', 'F25-2579', 'F25-2577', 'F24-3523', 'F25-0975', 'F25-1742', 'F25-0949', 'F25-1690', 'F25-1771', 'F24-3457', 'F24-3423', 'F25-0944', 'F24-3470', 'F25-0937', 'F24-3462', 'F25-0943', 'F24-3531', 'F24-3449', 'F26-0028', 'F24-3630', 'F24-3641', 'F24-3458', 'F26-0030', 'F24-3566', 'F24-3421', 'F25-2618', 'F25-0973', 'F24-3655', 'F25-1747', 'F24-3473', 'F24-3480', 'F24-3581', 'F25-0934', 'F25-0958', 'F24-3656', 'F25-0966', 'F25-0942', 'F25-1666', 'F24-3466', 'F24-3431', 'F25-1737', 'F24-3546', 'F25-1680', 'F26-0018', 'F25-1697', 'F25-1744', 'F24-3432', 'F25-1752', 'F24-3544', 'F24-3654', 'F25-1730

In [ ]:
commons_traits =  compiled[compiled['id_lab'].isin(commons)].copy()
print('commons_traits lenght:', len(commons_traits))
commons_traits.tail()

commons_traits lenght: 461


,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,net_gas_8h_ml,net_gas_24h_ml,gas_ml_g_dm_incubated_24h,tddm,part_fact,ch4_percentage_in_gas_8h,ch4_percentage_in_gas_24h,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h
663,F26-0038,CIAT-19165,Sesbania keniensis,Shrub_Trees,96.44,10.40,89.60,25.52,30.40,50.40,36.44,76.11,157.76,44.03,2.79,14.66,15.21,24.00,56.24,-8.23
664,F26-0039,CIAT-21899,Sesbania sesban,Shrub_Trees,95.84,7.99,92.01,18.32,41.37,63.03,36.09,68.90,143.71,36.07,2.51,14.06,14.81,21.29,61.07,0.43
665,F26-0040,CIAT-23414,Codariocalyx motorius,Shrub_Trees,96.31,9.34,90.66,16.19,43.82,62.85,29.68,59.26,122.99,26.23,2.13,14.60,15.26,18.76,73.03,-0.58
666,F26-0041,CIAT-23767,Desmodium nicaraguense,Shrub_Trees,95.00,10.10,89.90,19.05,26.90,54.72,39.44,74.68,157.14,46.73,2.97,16.24,16.82,26.44,56.97,1.72
667,F26-0042,CIAT-33127,Codariocalyx motorius,Shrub_Trees,95.60,9.53,90.47,13.66,45.27,64.59,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
legumes_traits_nirs = commons_traits.merge(leg_2, on='id_lab', how='left')
print('legumes_traits_nirs lenght:', len(legumes_traits_nirs))
legumes_traits_nirs.head(2)

legumes_traits_nirs lenght: 461


,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,...,2495.5,2496,2496.5,2497,2497.5,2498,2498.5,2499,2499.5,Unnamed: 4207
0,F24-3416,CIAT-705,Indigofera suffruticosa,Herbaceous_legumes,91.97,12.68,87.32,33.01,28.05,45.22,...,0.592270,0.592244,0.592214,0.592179,0.592138,0.592091,0.592035,0.591970,0.591889,NaN
1,F24-3417,CIAT-707,Alysicarpus ovalifolius,Herbaceous_legumes,90.09,12.07,87.93,25.40,25.44,54.11,...,0.493835,0.493839,0.493836,0.493827,0.493809,0.493783,0.493746,0.493696,0.493628,NaN


In [ ]:
legumes_traits_nirs.iloc[:2, 18:25]

,methane_intensity,ch4_ml_g_ndf_digested_24h,Unnamed: 6,400,400.5,401,401.5
0,31.84,149.32,NaN,0.983552,0.985116,0.986627,0.988091
1,36.44,125.18,NaN,0.941212,0.942474,0.943680,0.944833


In [ ]:
legumes_traits_nirs_clean = legumes_traits_nirs.drop(columns=legumes_traits_nirs.columns[legumes_traits_nirs.columns.str.contains('Unnamed')])

In [ ]:
legumes_traits_nirs_clean.head(2)

,id_lab,id,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,pc_percentage_dm,adf_percentage_dm,ndf_percentage_dm,...,2495,2495.5,2496,2496.5,2497,2497.5,2498,2498.5,2499,2499.5
0,F24-3416,CIAT-705,Indigofera suffruticosa,Herbaceous_legumes,91.97,12.68,87.32,33.01,28.05,45.22,...,0.592290,0.592270,0.592244,0.592214,0.592179,0.592138,0.592091,0.592035,0.591970,0.591889
1,F24-3417,CIAT-707,Alysicarpus ovalifolius,Herbaceous_legumes,90.09,12.07,87.93,25.40,25.44,54.11,...,0.493827,0.493835,0.493839,0.493836,0.493827,0.493809,0.493783,0.493746,0.493696,0.493628


In [ ]:
#legumes_traits_nirs_clean.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/07_nirs_compilation/legumes_nirs.csv', index = None)

# 8.0 Dashboard June 2026

Requested by: Alejandara Marín

Date: 2026_06_22

## 8.1. Data load

In [14]:
gas_av = pd.read_csv('/content/drive/MyDrive/lmf/data/2026_05_22_data_delivery/08_dashboard_june_2026/gas_clean_average_subsets_1234_2026_06_09.csv')

## 8.2 Filtering

In [18]:
gas_av.functional_group.unique()

array([nan, 'Herbaceous_legumes', 'Shrub_Trees', 'Browse', 'Grass',
       'Forage', 'Concentrate'], dtype=object)

In [21]:
grass = gas_av[gas_av['functional_group'] == 'Grass']
shrub = gas_av[gas_av['functional_group'] == 'Shrub_Trees']
legumes = gas_av[gas_av['functional_group'] == 'Herbaceous_legumes']

In [22]:
grass.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/grass.csv', index = None)
shrub.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/shrub_trees.csv', index = None)
legumes.to_csv('/content/drive/MyDrive/lmf/output/2026_05_22_data_delivery/08_dashboard_june_2026/legumes.csv', index = None)